# Lezione 07 — Test di ipotesi I

**Corso:** Statistica Applicata con Python · **Blocco 3:** Inferenza
**Tempo stimato:** 120 minuti · **Capitolo del manuale:** `manual/07_tests_hipotesis_i_IT.md`

## Cosa vedrai in questa lezione

- La **logica del test**: H0 e H1, statistica test, *p-value*, livello di significatività e regione di rifiuto.
- La **corretta interpretazione del p-value**, verificata con la simulazione.
- **Errori di tipo I e II** e **potenza** (a livello concettuale): come cambia con la dimensione del campione.
- **z-test e t-test a un campione**, **t-test per due campioni indipendenti (Welch)**, **t-test per campioni appaiati** e **test per due proporzioni**.
- La **dimensione dell'effetto** (differenza di medie e d di Cohen) insieme all'intervallo di confidenza.
- La **verifica dei presupposti** con i grafici.

## Prerequisiti

- Lezione 1 (variabilità campionaria), lezione 5 (distribuzione normale e *t*, teorema del limite centrale) e lezione 6 (errore standard e intervalli di confidenza).
- Librerie: `numpy`, `pandas`, `matplotlib`, `scipy`, `statsmodels` (`pip install numpy pandas matplotlib scipy statsmodels`; sono già installate in Google Colab).

> **Come usare questo notebook:** esegui le celle in ordine. Prima di ogni esperimento scrivi cosa ti aspetti che accada. Tutti i dati (tempi di caricamento, tassi di conversione) sono sintetici: un *esempio illustrativo*, non dati reali.

---
## Cella 0 — Setup

Importiamo le librerie e fissiamo il **seme**: con lo stesso seme, tutti otteniamo gli stessi numeri.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest

rng = np.random.default_rng(42)          # seme fisso: tutti otteniamo gli stessi risultati
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True
print("Librerie caricate correttamente.")

---
## 1. Il problema: la nuova versione della pagina è più veloce?

Un team web rilascia una nuova versione di una pagina. A 40 visite viene servita la versione vecchia e a 45 la nuova (ogni visita ne vede una sola). Misuriamo il **tempo di caricamento** in secondi. La domanda: la differenza che vediamo nelle medie è reale o può essere puro rumore campionario (lezione 1)?

In [ ]:
rng_datos = np.random.default_rng(31)
antigua = rng_datos.normal(3.2, 0.9, 40).clip(0.8)      # esempio illustrativo (s)
nueva   = rng_datos.normal(2.85, 0.9, 45).clip(0.8)

resumen = pd.DataFrame({
    "n": [len(antigua), len(nueva)],
    "media (s)": [antigua.mean(), nueva.mean()],
    "dev. standard (s)": [antigua.std(ddof=1), nueva.std(ddof=1)]},
    index=["versione vecchia", "versione nuova"])
display(resumen.round(3))

plt.boxplot([antigua, nueva], tick_labels=["vecchia", "nuova"])
plt.title("Tempo di caricamento per versione")
plt.ylabel("secondi")
plt.show()

---
## 2. La logica del test: H0, statistica test e p-value

- **H0 (ipotesi nulla):** non c'è nessuna differenza reale; la media della popolazione è la stessa nelle due versioni.
- **H1 (ipotesi alternativa):** le medie sono diverse.
- **Statistica test:** un numero che riassume quanto ciò che osserviamo si allontana da H0, misurato in errori standard.
- **p-value:** la probabilità di ottenere un risultato **almeno così estremo** come quello osservato **se H0 fosse vera**.

Calcoliamo a mano la statistica t di Welch (differenza / errore standard) e il suo p-value, e li confrontiamo con `scipy`.

In [ ]:
n1, n2 = len(antigua), len(nueva)
dif = antigua.mean() - nueva.mean()
v1, v2 = antigua.var(ddof=1) / n1, nueva.var(ddof=1) / n2
error_estandar = np.sqrt(v1 + v2)
t_obs = dif / error_estandar
gl = (v1 + v2) ** 2 / (v1 ** 2 / (n1 - 1) + v2 ** 2 / (n2 - 1))     # gradi di libertà di Welch
p_valor = 2 * stats.t.sf(abs(t_obs), gl)                              # due code

res = stats.ttest_ind(antigua, nueva, equal_var=False)
print(f"differenza tra le medie = {dif:.3f} s   errore standard = {error_estandar:.3f}")
print(f"a mano:  t = {t_obs:.3f}   gl = {gl:.1f}   p = {p_valor:.4f}")
print(f"scipy:  t = {res.statistic:.3f}   gl = {res.df:.1f}   p = {res.pvalue:.4f}")
assert np.isclose(t_obs, res.statistic) and np.isclose(p_valor, res.pvalue)

Con **α = 0,05** (due code) la **regione di rifiuto** è formata dai valori di t più estremi del valore critico: il 2,5 % di ciascuna coda sotto H0. Se il t osservato ci cade dentro, rifiutiamo H0.

In [ ]:
alfa = 0.05
t_crit = stats.t.ppf(1 - alfa / 2, gl)
x = np.linspace(-4, 4, 400)
y = stats.t.pdf(x, gl)
plt.plot(x, y, color="black")
plt.fill_between(x, y, where=(x <= -t_crit), color="tab:red", alpha=0.4, label="regione di rifiuto")
plt.fill_between(x, y, where=(x >= t_crit), color="tab:red", alpha=0.4)
plt.axvline(t_obs, color="tab:blue", linestyle="--", label=f"t osservato = {t_obs:.2f}")
plt.title("Distribuzione di t sotto H0 e regione di rifiuto (α = 0,05)")
plt.xlabel("t"); plt.legend()
plt.show()
print(f"valore critico = ±{t_crit:.3f}   →   si rifiuta H0?", abs(t_obs) > t_crit)

---
## 3. Una coda o due code

Se H1 è "la nuova è più veloce" (una direzione), il test è **a una coda**; se H1 è "sono diverse" (qualsiasi direzione), **a due code**. Una coda concentra tutto α su un lato e dà un p-value pari alla metà **se il segno è quello previsto**. Per questo la coda si sceglie **prima** di guardare i dati e in base alla domanda di business; farlo dopo, vedendo verso dove va la differenza, è barare.

In [ ]:
for alt, etiqueta in [("two-sided", "due code (H1: diverse)"),
                      ("greater", "una coda (H1: vecchia > nuova)"),
                      ("less", "una coda (H1: vecchia < nuova)")]:
    r = stats.ttest_ind(antigua, nueva, equal_var=False, alternative=alt)
    print(f"{etiqueta:<45} p = {r.pvalue:.4f}")

**Cosa osservi?**

- Il p-value a una coda nella direzione prevista è la metà di quello a due code; nella direzione opposta è quasi 1.
- Se avessimo deciso la coda *dopo* aver visto che la nuova versione è più veloce, che p-value avremmo ottenuto e quanto sarebbe rigoroso?

*(La tua risposta qui.)*

---
## 4. Esperimento guidato: cosa significa davvero il p-value

Simuliamo un mondo in cui **H0 è vera**: le due versioni hanno esattamente la stessa media (3,0 s). Ripetiamo 10 000 volte l'esperimento (stesse dimensioni, 40 e 45) e ogni volta calcoliamo il test di Welch. Domande: quale frazione di questi esperimenti dà un |t| almeno grande come quello osservato? Con che frequenza risulta "significativo" a α = 0,05?

In [ ]:
rng_sim = np.random.default_rng(101)
N_SIM = 10_000
g1 = rng_sim.normal(3.0, 0.9, (N_SIM, n1))
g2 = rng_sim.normal(3.0, 0.9, (N_SIM, n2))          # H0 vera: stessa media
sim = stats.ttest_ind(g1, g2, axis=1, equal_var=False)
p_sim = sim.pvalue

print(f"frazione con |t| ≥ |t osservato| = {(np.abs(sim.statistic) >= abs(t_obs)).mean():.4f}   (p del test = {res.pvalue:.4f})")
print(f"frazione 'significativa' con α = 0,05 = {(p_sim < 0.05).mean():.4f}")
print(f"frazione 'significativa' con α = 0,01 = {(p_sim < 0.01).mean():.4f}")

plt.hist(p_sim, bins=20, edgecolor="white")
plt.axhline(N_SIM / 20, color="black", linestyle="--")
plt.title("p-value di 10 000 esperimenti con H0 vera")
plt.xlabel("p-value"); plt.ylabel("numero di esperimenti")
plt.show()

**Cosa osservi?**

- La frazione con |t| ≥ |t osservato| è quasi uguale al p-value: è esattamente la sua definizione ("con che frequenza vedrei qualcosa del genere **se H0 fosse vera**?").
- Con H0 vera, circa il 5 % degli esperimenti risulta "significativo" a α = 0,05 e l'1 % a α = 0,01: **α è il tasso di falsi positivi** (errore di tipo I) quando H0 è vera.
- L'istogramma è quasi piatto: sotto H0 tutti i p-value sono ugualmente probabili. Un p = 0,03 non è "raro" se fai molti test (lo riprenderemo nella lezione 9).
- Questo esperimento risponde a "qual è la probabilità che H0 sia vera?" Perché no?

*(La tua risposta qui.)*

---
## 5. Errore di tipo II e potenza

| | H0 è vera | H1 è vera (c'è un effetto) |
|---|---|---|
| **Rifiuto H0** | Errore di tipo I (falso positivo), probabilità α | Decisione corretta, probabilità = **potenza** |
| **Non rifiuto H0** | Decisione corretta | Errore di tipo II (falso negativo), probabilità β = 1 − potenza |

La **potenza** è la probabilità di rilevare un effetto che **esiste**. Dipende dalla dimensione dell'effetto, dalla dimensione del campione, dalla variabilità e da α. Qui guardiamo solo come cambia con n; il **calcolo della dimensione campionaria** si fa nella lezione 13.

Simuliamo un effetto reale di 0,5 deviazioni standard tra due gruppi e facciamo variare la dimensione di ciascun gruppo.

In [ ]:
rng_pot = np.random.default_rng(43)
N_SIM_P = 4_000
EFECTO_REAL = 0.5            # in deviazioni standard (σ = 1)
filas = []
for n in [10, 20, 40, 80, 160]:
    x = rng_pot.normal(0, 1, (N_SIM_P, n))
    y = rng_pot.normal(EFECTO_REAL, 1, (N_SIM_P, n))
    potencia = (stats.ttest_ind(x, y, axis=1, equal_var=False).pvalue < 0.05).mean()
    filas.append({"n per gruppo": n, "potenza (simulata)": potencia,
                  "prob. errore di tipo II": 1 - potencia})
tabla_potencia = pd.DataFrame(filas).set_index("n per gruppo").round(3)
display(tabla_potencia)

plt.plot(tabla_potencia.index, tabla_potencia.iloc[:, 0], marker="o")
plt.axhline(0.8, color="black", linestyle="--")
plt.xscale("log")
plt.title("Potenza in base alla dimensione di ciascun gruppo (effetto = 0,5 σ, α = 0,05)")
plt.xlabel("n per gruppo (scala log)"); plt.ylabel("potenza")
plt.show()

**Cosa osservi?**

- Con 10 osservazioni per gruppo, con che frequenza rileviamo un effetto che **esiste**? Cosa significa un "non significativo" ottenuto con un campione piccolo?
- Quante osservazioni per gruppo servono, all'incirca, per arrivare all'80 % di potenza con questo effetto?
- Se l'effetto reale fosse la metà (0,25 σ), la potenza con lo stesso n sarebbe maggiore o minore?

*(La tua risposta qui.)*

---
## 6. Test a un campione: z-test e t-test

Otto misurazioni del tempo di una query (*esempio illustrativo*). L'obiettivo del team è una media di 3,0 s. H0: μ = 3,0; H1: μ ≠ 3,0.

- Se la deviazione standard della popolazione **σ fosse nota** (caso didattico, quasi mai reale) useremmo lo **z-test**: z = (x̄ − μ0) / (σ/√n), confrontato con la normale.
- In pratica σ si stima con s e si usa il **t-test**: t = (x̄ − μ0) / (s/√n), con n − 1 gradi di libertà.

In [ ]:
muestra = np.array([3.4, 2.9, 3.8, 3.1, 3.6, 3.3, 3.9, 3.2])
mu0 = 3.0
n = len(muestra)
media, s = muestra.mean(), muestra.std(ddof=1)

t_mano = (media - mu0) / (s / np.sqrt(n))
p_t = 2 * stats.t.sf(abs(t_mano), n - 1)
r1 = stats.ttest_1samp(muestra, mu0)
print(f"media = {media:.3f}   s = {s:.4f}   errore standard = {s / np.sqrt(n):.4f}")
print(f"t-test a mano: t = {t_mano:.3f}   p = {p_t:.4f}   |   scipy: t = {r1.statistic:.3f}   p = {r1.pvalue:.4f}")

SIGMA_CONOCIDA = 0.35        # ipotesi didattica
z = (media - mu0) / (SIGMA_CONOCIDA / np.sqrt(n))
p_z = 2 * stats.norm.sf(abs(z))
print(f"z-test (σ = {SIGMA_CONOCIDA}): z = {z:.3f}   p = {p_z:.4f}")

Nota che il valore di t con soli 7 gradi di libertà richiede più evidenza di quello di z: la distribuzione t ha code più pesanti per compensare il fatto che stimiamo σ con pochi dati (lezione 5).

---
## 7. Due campioni indipendenti: perché Welch

Il t-test classico di Student presuppone **varianze uguali** nei due gruppi. Quello di **Welch** (`equal_var=False`) non lo presuppone e corregge i gradi di libertà. Se le varianze sono simili, danno quasi lo stesso risultato; se non lo sono (soprattutto con dimensioni diverse), Student può sbagliare molto.

Esperimento: H0 vera (stessa media), ma un gruppo piccolo (n = 10) molto variabile (σ = 3) e uno grande (n = 50) poco variabile (σ = 1). Che tasso di falsi positivi dà ciascun test?

In [ ]:
rng_w = np.random.default_rng(202)
a = rng_w.normal(0, 3, (10_000, 10))
b = rng_w.normal(0, 1, (10_000, 50))          # H0 vera: entrambe con media 0
fp_student = (stats.ttest_ind(a, b, axis=1, equal_var=True).pvalue < 0.05).mean()
fp_welch = (stats.ttest_ind(a, b, axis=1, equal_var=False).pvalue < 0.05).mean()
print(f"Falsi positivi (α = 0,05):  Student = {fp_student:.3f}   Welch = {fp_welch:.3f}")

r_student = stats.ttest_ind(antigua, nueva, equal_var=True)
print(f"Con i dati del web (varianze simili):  Student p = {r_student.pvalue:.4f}   Welch p = {res.pvalue:.4f}")

**Cosa osservi?**

- Student, con varianze e dimensioni molto diverse, dà molti più "significativi" di quanti ne prometta α (qui più di 5 volte il 5 %). Welch resta vicino al 5 %.
- Con i dati del web la differenza tra i due è minima. Allora, cosa perdiamo usando sempre Welch? (Quasi nulla: per questo è la scelta prudente.)

*(La tua risposta qui.)*

---
## 8. Campioni appaiati

Ora misuriamo le **stesse 25 pagine** prima e dopo l'ottimizzazione. Le pagine sono molto diverse tra loro (alcune leggere, altre pesanti), ma ciascuna viene confrontata **con se stessa**. Il t-test per campioni appaiati (`ttest_rel`) lavora sulle **differenze** per pagina, eliminando così la variabilità tra pagine.

In [ ]:
rng_p = np.random.default_rng(5)
antes = rng_p.lognormal(1.1, 0.5, 25)                          # esempio illustrativo (s)
despues = antes - 0.20 + rng_p.normal(0, 0.25, 25)
difs = antes - despues

r_par = stats.ttest_rel(antes, despues)
r_ind = stats.ttest_ind(antes, despues, equal_var=False)       # SBAGLIATO per dati appaiati
r_dif = stats.ttest_1samp(difs, 0)
print(f"differenza media per pagina = {difs.mean():.3f} s   dev. standard delle differenze = {difs.std(ddof=1):.3f}")
print(f"Appaiato:    t = {r_par.statistic:.3f}   p = {r_par.pvalue:.2e}")
print(f"Indipendente (errato): t = {r_ind.statistic:.3f}   p = {r_ind.pvalue:.4f}")
print(f"Un campione sulle differenze: t = {r_dif.statistic:.3f}   p = {r_dif.pvalue:.2e}")

for i in range(25):
    plt.plot([0, 1], [antes[i], despues[i]], color="gray", alpha=0.6, marker="o")
plt.xticks([0, 1], ["prima", "dopo"])
plt.title("Tempo di caricamento delle stesse 25 pagine")
plt.ylabel("secondi")
plt.show()

**Cosa osservi?**

- Quasi tutte le linee scendono, ma le pagine partono da livelli molto diversi. Perché il test indipendente non vede l'effetto e quello appaiato sì?
- Il t-test appaiato è identico a un t-test a un campione sulle differenze (stesso t e stesso p).
- Quando **non** si può usare l'appaiato? (Quando i gruppi non hanno una corrispondenza uno a uno, come nella sezione 1, dove ogni visita vede una sola versione.)

*(La tua risposta qui.)*

---
## 9. Due proporzioni (tassi di conversione)

Un'altra metrica della pagina: il **tasso di conversione** (visite che acquistano). Gruppo vecchio: 200 conversioni su 4 000 visite (5,0 %); gruppo nuovo: 244 su 4 000 (6,1 %) (*esempio illustrativo*). H0: le due proporzioni sono uguali. È il test che userai nell'A/B (lezione 13).

`proportions_ztest` usa una deviazione standard **combinata** (sotto H0 le due proporzioni sono uguali, quindi i dati si uniscono). L'intervallo di confidenza della differenza usa invece l'errore standard **non combinato**.

In [ ]:
conversiones = np.array([244, 200])          # nuova, vecchia
visitas = np.array([4000, 4000])
z_prop, p_prop = proportions_ztest(conversiones, visitas)

p_nueva, p_antigua = conversiones / visitas
p_comb = conversiones.sum() / visitas.sum()
ee_comb = np.sqrt(p_comb * (1 - p_comb) * (1 / 4000 + 1 / 4000))
z_mano = (p_nueva - p_antigua) / ee_comb

ee_sin = np.sqrt(p_nueva * (1 - p_nueva) / 4000 + p_antigua * (1 - p_antigua) / 4000)
ic_dif = ((p_nueva - p_antigua) - 1.96 * ee_sin, (p_nueva - p_antigua) + 1.96 * ee_sin)

print(f"differenza tra le proporzioni = {p_nueva - p_antigua:.4f}   IC 95 % = ({ic_dif[0]:.4f}, {ic_dif[1]:.4f})")
print(f"statsmodels: z = {z_prop:.3f}   p = {p_prop:.4f}   |   a mano: z = {z_mano:.3f}")

# stessi tassi approssimativi con solo 400 visite per gruppo
z_peq, p_peq = proportions_ztest(np.array([26, 20]), np.array([400, 400]))
print(f"Con 400 visite per gruppo (6,5 % vs 5,0 %): z = {z_peq:.3f}   p = {p_peq:.4f}")

**Cosa osservi?**

- Con 4 000 visite per gruppo, la differenza di 1,1 punti percentuali risulta significativa, ma l'IC arriva quasi a 0. Cosa ti dice sulla precisione della stima?
- Con sole 400 visite per gruppo, una differenza maggiore (1,5 punti) **non** è significativa. Significa che non c'è effetto? (No: significa che non c'è evidenza sufficiente con quel campione.)

*(La tua risposta qui.)*

---
## 10. Dimensione dell'effetto: significativo non vuol dire importante

Il p-value dice se l'effetto è **compatibile con il caso**, non se è **grande**. Si riportano sempre:

- la **differenza di medie** nelle unità originali (secondi), con il suo **intervallo di confidenza** (lezione 6);
- la **d di Cohen**: la differenza di medie divisa per la deviazione standard combinata, cioè la differenza in "deviazioni standard". Si legge sempre **nel suo contesto**: 0,2 s possono essere irrilevanti in una pagina informativa e critici in un pagamento.

Applichiamo entrambe ai dati web e poi a un caso con n enorme.

In [ ]:
def d_de_cohen(a, b):
    """Differenza tra le medie / deviazione standard combinata."""
    na, nb = len(a), len(b)
    sp = np.sqrt(((na - 1) * np.var(a, ddof=1) + (nb - 1) * np.var(b, ddof=1)) / (na + nb - 2))
    return (np.mean(a) - np.mean(b)) / sp

ic = res.confidence_interval(confidence_level=0.95)
print(f"Dati web: differenza = {dif:.3f} s   IC 95 % = ({ic.low:.3f}, {ic.high:.3f})   d = {d_de_cohen(antigua, nueva):.2f}   p = {res.pvalue:.4f}")

rng_g = np.random.default_rng(303)
grande_a = rng_g.normal(3.0, 0.9, 20_000)
grande_b = rng_g.normal(2.96, 0.9, 20_000)
r_g = stats.ttest_ind(grande_a, grande_b, equal_var=False)
ic_g = r_g.confidence_interval(confidence_level=0.95)
print(f"n = 20 000 per gruppo: differenza = {grande_a.mean() - grande_b.mean():.3f} s   IC 95 % = ({ic_g.low:.3f}, {ic_g.high:.3f})   d = {d_de_cohen(grande_a, grande_b):.2f}   p = {r_g.pvalue:.1e}")

**Cosa osservi?**

- Il secondo caso è "molto significativo" (p minuscolo) ma il miglioramento è di circa 0,04 s, con d ≈ 0,04. Quel cambiamento giustifica il costo del rilascio? Dipende dal business, non dal p-value.
- Nei dati web, d ≈ 0,5 e un miglioramento di circa 0,4 s: più rilevante, ma con un IC piuttosto ampio (da circa 0,05 s a circa 0,75 s).

*(La tua risposta qui.)*

---
## 11. Verificare i presupposti

I t-test presuppongono dati approssimativamente normali (oppure n abbastanza grande perché la media lo sia, lezione 5), osservazioni indipendenti e, nel caso di Student, varianze uguali. La prima cosa è **guardare**: istogramma e grafico Q-Q. Un test di normalità (Shapiro-Wilk) è un complemento con dei **limiti**: con n piccolo quasi mai rifiuta (poca potenza) e con n grande rifiuta scostamenti senza importanza pratica.

In [ ]:
rng_n = np.random.default_rng(404)
asimetrica = rng_n.lognormal(0.9, 0.6, 30)        # esempio illustrativo: tempi con coda lunga

fig, ejes = plt.subplots(2, 2, figsize=(9, 6))
for fila, (datos, nombre) in enumerate([(nueva, "versione nuova (circa normale)"),
                                        (asimetrica, "campione asimmetrico")]):
    ejes[fila, 0].hist(datos, bins=10, edgecolor="white")
    ejes[fila, 0].set_title(nombre)
    stats.probplot(datos, dist="norm", plot=ejes[fila, 1])
    ejes[fila, 1].set_title("Grafico Q-Q")
plt.tight_layout()
plt.show()

print(f"Shapiro-Wilk versione nuova: p = {stats.shapiro(nueva).pvalue:.3f}")
print(f"Shapiro-Wilk campione asimmetrico (n = 30): p = {stats.shapiro(asimetrica).pvalue:.3f}   asimmetria = {stats.skew(asimetrica):.2f}")

rng_t = np.random.default_rng(505)
casi_normal = rng_t.standard_t(10, 5_000)
print(f"Shapiro-Wilk dati quasi normali ma n = 5 000: p = {stats.shapiro(casi_normal).pvalue:.1e}")

**Cosa osservi?**

- Il campione asimmetrico è chiaramente sbilanciato nell'istogramma e nel Q-Q, ma con n = 30 Shapiro-Wilk **non** rifiuta la normalità (p ≈ 0,10): il test ha poca potenza con n piccolo.
- Con n = 5 000 e dati quasi normali, Shapiro-Wilk rifiuta con un p minuscolo: con campioni grandi rileva scostamenti irrilevanti per un t-test.
- Morale: decidi con i grafici e il buon senso; se i dati sono molto asimmetrici o ci sono valori estremi e n è piccolo, la lezione 8 presenta i test non parametrici.

*(La tua risposta qui.)*

---
## Riepilogo della lezione

- Un test confronta ciò che osserviamo con quello che ci aspetteremmo **se H0 fosse vera**; il **p-value** è la probabilità di un risultato almeno così estremo sotto H0. **Non** è la probabilità che H0 sia vera né quella di aver sbagliato.
- **α** è il tasso di falsi positivi con H0 vera (lo abbiamo verificato: ≈ 5 % con α = 0,05). La **potenza** è la probabilità di rilevare un effetto reale e cresce con n.
- Una coda o due code si decide **prima** di vedere i dati.
- Strumenti: `ttest_1samp`, `ttest_ind(equal_var=False)` (Welch), `ttest_rel` (solo se c'è una corrispondenza uno a uno) e `proportions_ztest`.
- Riporta sempre **effetto + IC + p-value**. "Non significativo" non vuol dire "non c'è effetto"; "significativo" non vuol dire "importante".
- Ogni test in più che provi finché non trovi p < 0,05 gonfia i falsi positivi: la lezione 9 lo affronta (confronti multipli).

**Prossimo passo:** il notebook di pratica della lezione 7 e poi la lezione 8 (ANOVA, test non parametrici) e la lezione 9 (chi-quadro, confronti multipli).